# Semantic Similarity

In [1]:
!cd ../../.. && pip install -v . --group=dev --group=observability --group=analysis

Using pip 26.0.1 from /Users/mitjaorlov/.pyenv/versions/3.13.13/lib/python3.13/site-packages/pip (python 3.13)
Processing ./.
  Running command installing build dependencies
  Using pip 26.0.1 from /Users/mitjaorlov/.pyenv/versions/3.13.13/lib/python3.13/site-packages/pip (python 3.13)
    Obtaining dependency information for setuptools>=40.8.0 from https://files.pythonhosted.org/packages/5d/40/e1e72872c6354b306daef1703549e8e83b4d43cfea356311bf722a043752/setuptools-83.0.0-py3-none-any.whl.metadata
    Using cached setuptools-83.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached setuptools-83.0.0-py3-none-any.whl (1.0 MB)
  ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
  torch 2.12.0 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.
  Installing build dependencies ... done
  Running command Getting requirements to build wheel
  r

In [2]:
import warnings

In [3]:
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

In [4]:
from duckdb.sqltypes import VARCHAR
import duckdb as ddb

In [5]:
from sklearn.metrics.pairwise import cosine_similarity
COS_SIMILARITY_READY = False
try:
    # NOTE: this can not run natively on Intel mac anymore
    from sentence_transformers import SentenceTransformer
    COS_SIMILARITY_READY = True
except ImportError as e:
    COS_SIMILARITY_READY = False
    warnings.warn("FAILED TO IMPORT `sentence_transformers`")
    warnings.warn(str(e))
finally:
    print(f"{COS_SIMILARITY_READY=}")

COS_SIMILARITY_READY=True


In [6]:
import tol_colors as tcls

In [7]:
import itertools as itlx
import json
from pathlib import Path
import re 

In [8]:
from IPython.display import display, display_markdown

In [9]:
from pprint import pprint

## Data Loading

In [10]:
ddb.query("""
CREATE TABLE IF NOT EXISTS cons_dwc_t_data_nodes AS
    SELECT * FROM './procedural_data/dwc-t_data--cons_node_info_groups.csv';

CREATE TABLE IF NOT EXISTS cons_dwc_t_author_nodes AS
    SELECT * FROM './procedural_data/dwc-t_author--cons_node_info_groups.csv';

CREATE TABLE IF NOT EXISTS cons_eml_data_nodes AS
    SELECT * FROM './procedural_data/eml_data--cons_node_info_groups.csv';

CREATE TABLE IF NOT EXISTS cons_eml_author_nodes AS
    SELECT * FROM './procedural_data/eml_author--cons_node_info_groups.csv';
""")

In [11]:
ddb.query("""
CREATE TABLE IF NOT EXISTS gt_dwc_t_data_nodes AS
    SELECT * FROM './procedural_data/dwc-t_data--gt_node_info_groups.csv';

CREATE TABLE IF NOT EXISTS gt_dwc_t_author_nodes AS
    SELECT * FROM './procedural_data/dwc-t_author--gt_node_info_groups.csv';

CREATE TABLE IF NOT EXISTS gt_eml_data_nodes AS
    SELECT * FROM './procedural_data/eml_data--gt_node_info_groups.csv';

CREATE TABLE IF NOT EXISTS gt_eml_author_nodes AS
    SELECT * FROM './procedural_data/eml_author--gt_node_info_groups.csv';
""")

In [12]:
ddb.query("""
DESCRIBE gt_dwc_t_data_nodes;
""")

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ column00         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ case_id          │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ tag              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ depth            │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ degree           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_leaf          │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ attributes       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ attributes_count │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ has_attributes   │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ text             │ VARC

## By Shared Ancestor Paths

In [13]:
ddb.query("""
SELECT c.path, c.text, g.text, c.case_id
FROM cons_dwc_t_data_nodes AS c 
JOIN gt_dwc_t_data_nodes AS g
    ON  c.path = g.path 
    AND c.case_id = g.case_id
""")

┌───────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────┬───────────┬─────────┐
│                                                     path                                                      │      text      │   text    │ case_id │
│                                                    varchar                                                    │    varchar     │  varchar  │  int64  │
├───────────────────────────────────────────────────────────────────────────────────────────────────────────────┼────────────────┼───────────┼─────────┤
│ .                                                                                                             │ \n             │ \n        │       0 │
│ {http://rs.tdwg.org/dwc/text/}core                                                                            │ \n             │ \n        │       0 │
│ {http://rs.tdwg.org/dwc/text/}core/{http://rs.tdwg.org/dwc/text/}files          

In [14]:
ddb.query("""
SELECT c.case_id, c.path, c.attributes, g.attributes
FROM cons_dwc_t_data_nodes AS c 
JOIN gt_dwc_t_data_nodes AS g
    ON  c.path = g.path 
    AND c.case_id = g.case_id
""")

┌─────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ case_id │                                                     path                                                      │                                                                            attributes                                                                            │                                                                                    attributes                                                                                     │
│  int64  │                                                    var

In [15]:
if COS_SIMILARITY_READY:
    sentence_transformer_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [16]:
def compute_cos_similarity(v1: str, v2:str) -> float | None:
    if v1 == v2:
        return 1.0
    if not v1 or not v2:
        return None
    
    emb1 = sentence_transformer_model.encode(v1).reshape(1,-1)
    emb2 = sentence_transformer_model.encode(v2).reshape(1,-1)

    return cosine_similarity(emb1, emb2)[0][0].item()

In [17]:
compute_cos_similarity("{'index': '21', 'term': 'http://rs.tdwg.org/dwc/terms/kingdom'}", "{'index': '21', 'term': 'http://rs.tdwg.org/dwc/terms/kingdom'}")

1.0

In [18]:
ddb.create_function("cos_sim", compute_cos_similarity)

In [19]:
def wrap_text_struct(v: str) -> str:
    return json.dumps({"text": v})

In [20]:
ddb.create_function("wrap_text_struct", wrap_text_struct)

In [34]:
%%time
cons_gt_dwc_t_data_nodes_cos_sim = ddb.query("""
SELECT 
    c.case_id, c.path, 
    c.text as cons_t, g.text as gt_t, 
    wrap_text_struct(ifnull(c.text, '')) as cons_wrap_t, wrap_text_struct(ifnull(g.text, '')) as gt_wrap_t, 
    c.attributes as cons_attributes, g.attributes as gt_attributes, 
    cos_sim(c.attributes, g.attributes) AS cos_sim_attrs, 
    cos_sim(wrap_text_struct(cons_wrap_t), wrap_text_struct(gt_wrap_t)) AS cos_sim_text
FROM cons_dwc_t_data_nodes AS c 
JOIN gt_dwc_t_data_nodes AS g
    ON  c.path = g.path 
    AND c.case_id = g.case_id
""").to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 38.9 s, sys: 12.4 s, total: 51.3 s
Wall time: 35 s


In [35]:
cons_gt_dwc_t_data_nodes_cos_sim.describe(include="all")

,case_id,path,cons_t,gt_t,cons_wrap_t,gt_wrap_t,cons_attributes,gt_attributes,cos_sim_attrs,cos_sim_text
count,8283.000000,8283,975,966,8283,8283,8283,8283,8283.000000,8283.000000
unique,NaN,332,24,29,25,30,2868,2808,NaN,NaN
top,NaN,.,\n,\n,"{""text"": """"}","{""text"": """"}",{},{},NaN,NaN
freq,NaN,174,267,260,7308,7317,534,537,NaN,NaN
mean,88.857781,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.933697,0.998633
std,51.662181,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.115382,0.015778
min,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.126419,0.692121
25%,43.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.885947,1.000000
50%,91.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000
75%,134.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000


In [36]:
cons_gt_dwc_t_data_nodes_cos_sim

,case_id,path,cons_t,gt_t,cons_wrap_t,gt_wrap_t,cons_attributes,gt_attributes,cos_sim_attrs,cos_sim_text
0,0,.,\n,\n,"{""text"": ""\n ""}","{""text"": ""\n ""}",{'metadata': 'meta.xml'},{'metadata': 'eml.xml'},0.773793,1.000000
1,0,{http://rs.tdwg.org/dwc/text/}core,\n,\n,"{""text"": ""\n ""}","{""text"": ""\n ""}",{'rowType': 'http://rs.tdwg.org/dwc/terms/Occu...,"{'encoding': 'UTF-8', 'fieldsTerminatedBy': '\...",0.890905,1.000000
2,0,{http://rs.tdwg.org/dwc/text/}core/{http://rs....,\n,\n,"{""text"": ""\n ""}","{""text"": ""\n ""}",{},{},1.000000,1.000000
3,0,{http://rs.tdwg.org/dwc/text/}core/{http://rs....,occurrence.txt,event.txt,"{""text"": ""occurrence.txt""}","{""text"": ""event.txt""}",{},{},1.000000,0.855316
4,0,{http://rs.tdwg.org/dwc/text/}core/{http://rs....,NaN,NaN,"{""text"": """"}","{""text"": """"}",{'index': '0'},{'index': '0'},1.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...
8278,99,{http://rs.tdwg.org/dwc/text/}extension/{http:...,NaN,NaN,"{""text"": """"}","{""text"": """"}","{'index': '26', 'term': 'http://rs.tdwg.org/dw...","{'index': '26', 'term': 'http://rs.tdwg.org/dw...",1.000000,1.000000
8279,99,{http://rs.tdwg.org/dwc/text/}extension/{http:...,NaN,NaN,"{""text"": """"}","{""text"": """"}","{'index': '27', 'term': 'http://rs.tdwg.org/dw...","{'index': '27', 'term': 'http://rs.tdwg.org/dw...",1.000000,1.000000
8280,99,{http://rs.tdwg.org/dwc/text/}extension/{http:...,NaN,NaN,"{""text"": """"}","{""text"": """"}","{'index': '28', 'term': 'http://rs.tdwg.org/dw...","{'index': '28', 'term': 'http://rs.tdwg.org/dw...",1.000000,1.000000
8281,99,{http://rs.tdwg.org/dwc/text/}extension/{http:...,NaN,NaN,"{""text"": """"}","{""text"": """"}","{'index': '29', 'term': 'http://rs.tdwg.org/dw...","{'index': '29', 'term': 'http://rs.tdwg.org/dw...",1.000000,1.000000


In [37]:
%%time
cons_gt_dwc_t_author_nodes_cos_sim = ddb.query("""
SELECT 
    c.case_id, c.path, 
    c.text as cons_t, g.text as gt_t, 
    wrap_text_struct(ifnull(c.text, '')) as cons_wrap_t, wrap_text_struct(ifnull(g.text, '')) as gt_wrap_t, 
    c.attributes as cons_attributes, g.attributes as gt_attributes, 
    cos_sim(c.attributes, g.attributes) AS cos_sim_attrs, 
    cos_sim(wrap_text_struct(cons_wrap_t), wrap_text_struct(gt_wrap_t)) AS cos_sim_text
FROM cons_dwc_t_author_nodes AS c 
JOIN gt_dwc_t_author_nodes AS g
    ON  c.path = g.path 
    AND c.case_id = g.case_id
""").to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 44.1 s, sys: 14.4 s, total: 58.5 s
Wall time: 39.8 s


In [38]:
%%time
cons_gt_eml_data_nodes_cos_sim = ddb.query("""
SELECT 
    c.case_id, c.path, 
    c.text as cons_t, g.text as gt_t, 
    wrap_text_struct(ifnull(c.text, '')) as cons_wrap_t, wrap_text_struct(ifnull(g.text, '')) as gt_wrap_t, 
    c.attributes as cons_attributes, g.attributes as gt_attributes, 
    cos_sim(c.attributes, g.attributes) AS cos_sim_attrs, 
    cos_sim(wrap_text_struct(cons_wrap_t), wrap_text_struct(gt_wrap_t)) AS cos_sim_text
FROM cons_eml_data_nodes AS c 
JOIN gt_eml_data_nodes AS g
    ON  c.path = g.path 
    AND c.case_id = g.case_id
""").to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 19.1 s, sys: 3.25 s, total: 22.4 s
Wall time: 33.6 s


In [39]:
%%time
cons_gt_eml_author_nodes_cos_sim = ddb.query("""
SELECT 
    c.case_id, c.path, 
    c.text as cons_t, g.text as gt_t, 
    wrap_text_struct(ifnull(c.text, '')) as cons_wrap_t, wrap_text_struct(ifnull(g.text, '')) as gt_wrap_t, 
    c.attributes as cons_attributes, g.attributes as gt_attributes, 
    cos_sim(c.attributes, g.attributes) AS cos_sim_attrs, 
    cos_sim(wrap_text_struct(cons_wrap_t), wrap_text_struct(gt_wrap_t)) AS cos_sim_text
FROM cons_eml_author_nodes AS c 
JOIN gt_eml_author_nodes AS g
    ON  c.path = g.path 
    AND c.case_id = g.case_id
""").to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

CPU times: user 18.4 s, sys: 3.06 s, total: 21.5 s
Wall time: 33.9 s


In [43]:
cons_gt_dwc_t_data_nodes_cos_sim.describe(include="all")

,case_id,path,cons_t,gt_t,cons_wrap_t,gt_wrap_t,cons_attributes,gt_attributes,cos_sim_attrs,cos_sim_text
count,8283.000000,8283,975,966,8283,8283,8283,8283,8283.000000,8283.000000
unique,NaN,332,24,29,25,30,2868,2808,NaN,NaN
top,NaN,.,\n,\n,"{""text"": """"}","{""text"": """"}",{},{},NaN,NaN
freq,NaN,174,267,260,7308,7317,534,537,NaN,NaN
mean,88.857781,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.933697,0.998633
std,51.662181,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.115382,0.015778
min,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.126419,0.692121
25%,43.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.885947,1.000000
50%,91.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000
75%,134.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000


In [44]:
cons_gt_dwc_t_author_nodes_cos_sim.describe(include="all")

,case_id,path,cons_t,gt_t,cons_wrap_t,gt_wrap_t,cons_attributes,gt_attributes,cos_sim_attrs,cos_sim_text
count,8254.000000,8254,984,975,8254,8254,8254,8254,8254.000000,8254.000000
unique,NaN,330,23,28,24,29,2829,2762,NaN,NaN
top,NaN,.,\n,\n,"{""text"": """"}","{""text"": """"}",{},{},NaN,NaN
freq,NaN,174,270,263,7270,7279,540,543,NaN,NaN
mean,89.336079,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.924626,0.998220
std,50.617869,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.122398,0.018201
min,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.136131,0.692121
25%,46.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.842089,1.000000
50%,91.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000
75%,133.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000


In [45]:
cons_gt_eml_data_nodes_cos_sim.describe(include="all")

,case_id,path,cons_t,gt_t,cons_wrap_t,gt_wrap_t,cons_attributes,gt_attributes,cos_sim_attrs,cos_sim_text
count,2844.000000,2844,2797,2688,2844,2844,2844,2844,2844.000000,2844.000000
unique,NaN,106,403,636,404,637,80,206,NaN,NaN
top,NaN,.,\n,\n,"{""text"": ""\n ""}","{""text"": ""\n ""}",{},{},NaN,NaN
freq,NaN,150,539,308,539,308,2649,2466,NaN,NaN
mean,85.343530,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.924282,0.867730
std,53.154125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.202199,0.205442
min,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.153541,0.015169
25%,40.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.732458
50%,83.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000
75%,132.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000


In [46]:
cons_gt_eml_author_nodes_cos_sim.describe(include="all")

,case_id,path,cons_t,gt_t,cons_wrap_t,gt_wrap_t,cons_attributes,gt_attributes,cos_sim_attrs,cos_sim_text
count,2760.000000,2760,2703,2609,2760,2760,2760,2760,2760.000000,2760.000000
unique,NaN,68,561,628,562,629,87,206,NaN,NaN
top,NaN,.,\n,\n,"{""text"": ""\n ""}","{""text"": ""\n ""}",{},{},NaN,NaN
freq,NaN,148,514,286,514,286,2568,2387,NaN,NaN
mean,90.430435,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.922567,0.880903
std,51.683751,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.204580,0.188933
min,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.120994,0.026708
25%,47.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.747712
50%,89.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000
75%,139.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,1.000000


In [47]:
cons_gt_eml_author_nodes_cos_sim.groupby(by="case_id").describe(include="all")

path                                                ... cos_sim_text  \
        count unique top freq mean  std  min  25%  50%  75%  ...       unique   
case_id                                                      ...                
0          31     31   .    1  NaN  NaN  NaN  NaN  NaN  NaN  ...          NaN   
3          23     23   .    1  NaN  NaN  NaN  NaN  NaN  NaN  ...          NaN   
4          32     32   .    1  NaN  NaN  NaN  NaN  NaN  NaN  ...          NaN   
5          23     23   .    1  NaN  NaN  NaN  NaN  NaN  NaN  ...          NaN   
6          26     26   .    1  NaN  NaN  NaN  NaN  NaN  NaN  ...          NaN   
...       ...    ...  ..  ...  ...  ...  ...  ...  ...  ...  ...          ...   
172        16     16   .    1  NaN  NaN  NaN  NaN  NaN  NaN  ...          NaN   
174        22     22   .    1  NaN  NaN  NaN  NaN  NaN  NaN  ...          NaN   
175         9      9   .    1  NaN  NaN  NaN  NaN  NaN  NaN  ...          NaN   
176        15     15   .    1  NaN  NaN  NaN  NaN  NaN  NaN  ...          NaN   
178        13     13   .    1  NaN  NaN  NaN  NaN  NaN  NaN  ...          NaN   

                                                                              
        top freq      mean       std       min       25%       50%  75%  max  
case_id                                                                       
0       NaN  NaN  0.816442  0.255191  0.103202  0.722639  1.000000  1.0  1.0  
3       NaN  NaN  0.890193  0.166120  0.427052  0.747188  1.000000  1.0  1.0  
4       NaN  NaN  0.853356  0.205620  0.173113  0.698017  1.000000  1.0  1.0  
5       NaN  NaN  0.857217  0.209014  0.167688  0.747626  0.966109  1.0  1.0  
6       NaN  NaN  0.901448  0.178618  0.308761  0.842704  1.000000  1.0  1.0  
...      ..  ...       ...       ...       ...       ...       ...  ...  ...  
172     NaN  NaN  0.942183  0.105873  0.690260  0.949233  1.000000  1.0  1.0  
174     NaN  NaN  0.851285  0.249750  0.067238  0.745305  0.998818  1.0  1.0  
175     NaN  NaN  0.902480  0.160690  0.578226  0.865086  1.000000  1.0  1.0  
176     NaN  NaN  0.853665  0.211293  0.401074  0.733066  1.000000  1.0  1.0  
178     NaN  NaN  0.852531  0.277483  0.048118  0.775051  1.000000  1.0  1.0  

[148 rows x 99 columns]